# 🚀 Тестирование Hunyuan3D-2.1 (Optimized for 8GB)
## 🎯 Цель ноутбука

Оценка производительности и качества обновленной модели **Hunyuan3D-2.1** на видеокартах с ограниченным объемом VRAM (8 ГБ). 

В рамках исследования проверяются:
1. Качество генерации геометрии с использованием **DiT Flow Matching**.
2. Эффективность текстурирования в режиме PBR.
3. Стабильность работы на 8 ГБ VRAM благодаря оптимизациям от IgorAherne.
4. Сравнение с результатами первой версии (Hunyuan3D-1).


## 🛠 1. Настройка окружения

In [ ]:
import os
import sys
import torch
import gc
import glob
import time
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display, HTML

NOTEBOOK_DIR = os.getcwd()
BASE_ML = os.path.abspath(os.path.join(NOTEBOOK_DIR, "..", ".."))
H3D2_PATH = os.path.join(BASE_ML, "H3D2")
OUTPUT_BASE_DIR = os.path.join(NOTEBOOK_DIR, "output")
SRC_IMAGES_DIR = os.path.join(BASE_ML, "src")

if H3D2_PATH not in sys.path:
    sys.path.insert(0, H3D2_PATH)
    sys.path.insert(0, os.path.join(H3D2_PATH, "hy3dshape"))
    sys.path.insert(0, os.path.join(H3D2_PATH, "hy3dpaint"))

os.chdir(H3D2_PATH)

from hy3dshape.rembg import BackgroundRemover
from hy3dshape.pipelines import Hunyuan3DDiTFlowMatchingPipeline
from textureGenPipeline import Hunyuan3DPaintPipeline, Hunyuan3DPaintConfig

try:
    from torchvision_fix import apply_fix
    apply_fix()
except ImportError:
    print("Warning: torchvision_fix module not found")

def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

print(f"Готов к работе. Проект: {H3D2_PATH}")
clear_gpu()

## 🧠 2. Инициализация моделей

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_PATH = 'tencent/Hunyuan3D-2.1'

print("Загрузка Shape Pipeline (DiT)...")
pipeline_shapegen = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(MODEL_PATH)
pipeline_shapegen.to(DEVICE)

print("Загрузка Paint Pipeline (PBR)...")
max_num_view = 6 
resolution = 512
conf = Hunyuan3DPaintConfig(max_num_view, resolution)
conf.realesrgan_ckpt_path = os.path.join(H3D2_PATH, "hy3dpaint/ckpt/RealESRGAN_x4plus.pth")
conf.multiview_cfg_path = os.path.join(H3D2_PATH, "hy3dpaint/cfgs/hunyuan-paint-pbr.yaml")
conf.custom_pipeline = os.path.join(H3D2_PATH, "hy3dpaint/hunyuanpaintpbr")
paint_pipeline = Hunyuan3DPaintPipeline(conf)

rembg = BackgroundRemover()

print("Модели загружены.")
clear_gpu()

## 🚀 3. Запуск генерации

In [ ]:
def run_generation(image_path):
    file_name = os.path.basename(image_path)
    output_name = os.path.splitext(file_name)[0]
    run_output_dir = os.path.join(OUTPUT_BASE_DIR, output_name)
    os.makedirs(run_output_dir, exist_ok=True)
    
    # 1. Загрузка и удаление фона
    input_img = Image.open(image_path).convert("RGBA")
    rgba_img = rembg(input_img)
    rgba_img.save(os.path.join(run_output_dir, "rgba.png"))
    
    # 2. Генерация геометрии (.glb)
    print(f"Генерация меша для {file_name}...")
    clear_gpu()
    mesh = pipeline_shapegen(image=rgba_img)[0]
    mesh_path = os.path.join(run_output_dir, "shape.glb")
    mesh.export(mesh_path)
    
    # 3. Текстурирование (PBR)
    print(f"Текстурирование {file_name}...")
    clear_gpu()
    output_mesh_path = os.path.join(run_output_dir, "textured.glb")
    paint_pipeline(
        mesh_path = mesh_path, 
        image_path = image_path,
        output_mesh_path = output_mesh_path
    )
    
    print(f"✅ {file_name} успешно обработан")
    clear_gpu()

image_files = []
for ext in ['*.png', '*.jpg', '*.jpeg', '*.webp']:
    image_files.extend(glob.glob(os.path.join(SRC_IMAGES_DIR, ext)))

print(f"Начинаю обработку {len(image_files)} файлов...")

for img_path in sorted(image_files):
    try:
        start_time = time.time()
        run_generation(img_path)
        print(f"Время выполнения: {time.time() - start_time:.2f} сек.")
    except Exception as e:
        print(f"❌ Ошибка в {img_path}: {e}")
        clear_gpu()

## 📊 Анализ результатов

Hunyuan3D-2.1 демонстрирует значительный прогресс по сравнению с первой версией:
1. **Геометрия**: Более детализированные меши благодаря Flow Matching.
2. **Текстуры**: PBR-пайплайн дает более реалистичные материалы.
3. **VRAM**: Оптимизации позволяют избежать OOM на 8 ГБ, хотя процесс остается ресурсоемким.
